# End-to-End ETL Pipeline (Local)

Runs the full WeatherSpeak PH pipeline locally using `modal_etl/core/` modules.

**Pipeline:** PDF → OCR markdown + metadata → radio scripts + TTS text → MP3

The same `run_step1 / run_step2 / run_step3` functions are used by the Modal ETL in production.
Output mirrors the Modal Volume layout: `output/{stem}/ocr.md`, `radio_{lang}.md`, etc.

Set `force=True` in any step cell to re-run that step even if outputs already exist.

In [9]:
import sys
from pathlib import Path

# Make modal_etl importable from notebook directory
sys.path.insert(0, str(Path.cwd().parent))

from modal_etl.core.ocr import run_step1
from modal_etl.core.scripts import run_step2
from modal_etl.core.tts import run_step3

In [10]:
# ── Configuration ────────────────────────────────────────────────────────
STEM          = "PAGASA_25-TC22_Verbena_TCB#24"
PDF_PATH      = Path("../data/bulletin-archive/archive/pagasa-25-TC22") / f"{STEM}.pdf"
OUTPUT_DIR    = Path("10-etl-e2e/output")
OLLAMA_URL    = "http://localhost:11434"
LANGUAGES     = ["ceb", "tl", "en"]
TTS_MODELS_DIR = Path.home() / ".cache" / "huggingface" / "hub"
FORCE_RUN     = True
BACKEND       = "marker"   # swap to "marker" to use Marker OCR or "gemma4" to use Gemma4 OCR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF:        {PDF_PATH}  (exists={PDF_PATH.exists()})")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Ollama URL: {OLLAMA_URL}")
print(f"Backend:    {BACKEND}")

PDF:        ../data/bulletin-archive/archive/pagasa-25-TC22/PAGASA_25-TC22_Verbena_TCB#24.pdf  (exists=True)
Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output
Ollama URL: http://localhost:11434
Backend:    marker


## Step 1: OCR — PDF → Markdown + Metadata

Sends each PDF page to Gemma 4 E4B via Ollama vision API.
Writes `ocr.md`, `chart.png`, and `metadata.json` to `output/{stem}/`.

In [ ]:
stem_dir = run_step1(PDF_PATH, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN, backend=BACKEND)
print(f"\nStep 1 complete → {stem_dir}")

# Preview OCR markdown
ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
print(f"\n--- ocr.md preview (first 500 chars) ---")
print(ocr_md[:500])

# Pretty-print metadata
import json
metadata = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"\n--- metadata.json ---")
print(json.dumps(metadata, indent=2, ensure_ascii=False)[:1000])

## Step 1 Validation

Runs OCR + metadata for the bulletin configured in `STEM` above, then checks outputs.

In [12]:
import json

stem_dir = OUTPUT_DIR / STEM

# Check ocr.md
ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
table_leaked = any(f"{h}-Hour Forecast" in ocr_md for h in [12, 24, 36, 48, 60, 72, 96, 120])
print(f"ocr.md: {len(ocr_md)} chars | forecast table leaked in narrative: {table_leaked}")

# forecast_table.md — only produced by gemma4 backend
ft_path = stem_dir / "forecast_table.md"
if ft_path.exists():
    print(f"forecast_table.md: {len(ft_path.read_text(encoding='utf-8'))} chars")
else:
    print("forecast_table.md: absent (expected in marker mode)")

# Check metadata.json
meta = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"wind_extent:        {meta.get('wind_extent')}")
print(f"land_hazards:       {meta.get('land_hazards')}")
print(f"track_outlook:      {meta.get('track_outlook')}")
print(f"forecast_positions: {len(meta.get('forecast_positions', []))} rows")


## Step 2: Radio Scripts + TTS Text

Generates a spoken weather announcement and TTS-optimised plain text for each language.
Writes `radio_{lang}.md` and `tts_{lang}.txt` to `output/{stem}/`.

In [ ]:
for lang in LANGUAGES:
    radio_path = run_step2(STEM, lang, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
    print(f"\n{'='*60}")
    print(f"[{lang.upper()}] {radio_path.name}")
    print('='*60)
    print(radio_path.read_text(encoding="utf-8")[:400])
print("\nStep 2 complete.")

## Step 3: TTS Synthesis → MP3

Synthesizes MP3 audio for each language using:
- English: Coqui XTTS v2 (speaker: Damien Black)
- Tagalog / Cebuano: Facebook MMS VITS

Writes `audio_{lang}.mp3` to `output/{stem}/`.

In [ ]:
for lang in LANGUAGES:
    mp3_path = run_step3(STEM, lang, OUTPUT_DIR, TTS_MODELS_DIR, force=FORCE_RUN)
    size_kb = mp3_path.stat().st_size // 1024
    print(f"[{lang.upper()}] {mp3_path.name}  ({size_kb} KB)")
print("\nStep 3 complete.")

In [ ]:
from IPython.display import Audio, display, Markdown

for lang in LANGUAGES:
    mp3_path = OUTPUT_DIR / STEM / f"audio_{lang}.mp3"
    if mp3_path.exists():
        display(Markdown(f"**{lang.upper()}**"))
        display(Audio(str(mp3_path)))

## Output Summary

In [ ]:
stem_dir = OUTPUT_DIR / STEM
print(f"Artefacts in {stem_dir.resolve()}:\n")
for f in sorted(stem_dir.iterdir()):
    size = f.stat().st_size
    unit = 'KB' if size >= 1024 else 'B'
    val = size // 1024 if size >= 1024 else size
    print(f"  {f.name:<35}  {val:>6} {unit}")